# Parsers against each other and against the judge

Runs the current Python parser, the Scala `PeriodParser` (through `PeriodParserCli` and sbt) and
`period_new` over every string in `data/periods.jsonl`, shows where they disagree, and scores each
against the judge's answers in `data/judge-output`. Ranges are compared as inclusive day bounds
with open sides as `None`; identifiers are not compared.

In [1]:
import json
import subprocess
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

from adapters.transformers.ebsco.production import _parse_period_or_bare_label
from adapters.transformers.ebsco.label_subdivisions import build_concept
from adapters.transformers.marc.parsers.period_new import parse as parse_new
from adapters.transformers.utils.text_utils import normalise_label

rows = [json.loads(line) for line in Path("data/periods.jsonl").open(encoding="utf-8")]
occurrences = Counter()
for row in rows:
    occurrences[(row["path"], row["text"])] += row["n"]
print(f"{sum(occurrences.values())} strings, {len(occurrences)} distinct (path, text) pairs")

1550106 strings, 75293 distinct (path, text) pairs


In [2]:
OPEN = {"0001-01-01", "9999-12-31", "-9999-01-01"}


def bounds(start, end):
    return tuple(None if not d or d[:10] in OPEN else d[:10] for d in (start, end))


def python_bounds(path, text):
    try:
        concept = _parse_period_or_bare_label(normalise_label(text, "Period")) if path == "production" else build_concept(text, "Period")
    except Exception:
        return (None, None)
    rng = getattr(concept, "range", None)
    return bounds(rng.from_time, rng.to_time) if rng else (None, None)


def new_bounds(path, text):
    span = parse_new(text)
    return bounds(span[0].isoformat(), span[1].isoformat()) if span else (None, None)

In [3]:
scala_in = Path("data/scala-in.txt").resolve()
scala_out = Path("data/scala-out.jsonl").resolve()
scala_in.write_text("\n".join(sorted({text.replace("\n", " ") for _, text in occurrences})), encoding="utf-8")
sbt = subprocess.run(
    ["sbt", "-batch", f"transformer_common/Test/runMain weco.pipeline.transformer.parse.PeriodParserCli {scala_in} {scala_out}"],
    cwd="../..", capture_output=True, text=True,
)
assert sbt.returncode == 0, sbt.stdout[-2000:] + sbt.stderr[-2000:]

scala = {}
for line in scala_out.read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    rng = row["range"] or {}
    scala[row["input"]] = bounds(rng.get("from"), rng.get("to"))

PARSERS = {"python": python_bounds, "scala": lambda path, text: scala[text], "new": new_bounds}
results = {name: {key: parser(*key) for key in occurrences} for name, parser in PARSERS.items()}

In [4]:
def verdict(key):
    got = {name: results[name][key] for name in PARSERS}
    if len(set(got.values())) == 1:
        return "all agree"
    return "differ: " + ", ".join(sorted(name for name in PARSERS if list(got.values()).count(got[name]) == 1))


verdicts = {key: verdict(key) for key in occurrences}
summary = Counter()
for key, v in verdicts.items():
    summary[(key[0], v)] += occurrences[key]
for (path, v), n in sorted(summary.items()):
    print(f"{path:11} {v:24} {n:>9}")

genre       all agree                     7407
genre       differ: new                      2
genre       differ: new, python, scala         4
genre       differ: python                 954
production  all agree                  1260619
production  differ: new                    616
production  differ: new, python, scala      6742
production  differ: python               74250
production  differ: scala                28902
subject     all agree                   135278
subject     differ: new                  35092
subject     differ: new, python, scala        75
subject     differ: python                 154
subject     differ: scala                   11


In [5]:
LIMIT = 40


def show(b):
    return f"{b[0] or 'open'} .. {b[1] or 'open'}" if b != (None, None) else "no range"


for key in sorted((k for k in occurrences if verdicts[k] != "all agree"), key=lambda k: -occurrences[k])[:LIMIT]:
    path, text = key
    print(f"{text!r}  [{path}, {occurrences[key]}x]")
    for name in PARSERS:
        print(f"    {name:7} {show(results[name][key])}")

'19th-20th centuries.'  [subject, 6934x]
    python  no range
    scala   no range
    new     1800-01-01 .. 1999-12-31
'18th-19th centuries.'  [subject, 2577x]
    python  no range
    scala   no range
    new     1700-01-01 .. 1899-12-31
'To 1500.'  [subject, 1785x]
    python  no range
    scala   no range
    new     open .. 1500-12-31
'Revolution, 1775-1783'  [subject, 1510x]
    python  no range
    scala   no range
    new     1775-01-01 .. 1783-12-31
'16th-17th centuries.'  [subject, 1196x]
    python  no range
    scala   no range
    new     1500-01-01 .. 1699-12-31
'17th-18th centuries.'  [subject, 1117x]
    python  no range
    scala   no range
    new     1600-01-01 .. 1799-12-31
'18th-20th centuries.'  [subject, 891x]
    python  no range
    scala   no range
    new     1700-01-01 .. 1999-12-31
'Revolution, 1775-1783.'  [subject, 823x]
    python  no range
    scala   no range
    new     1775-01-01 .. 1783-12-31
'Revolution, 1789-1799.'  [subject, 817x]
    python  no 

## Against the judge

`unparseable` and `ambiguous` both mean no range is right. Judge answers that are malformed or
run backwards are dropped. Precision is correct ranges over ranges produced; recall is correct
ranges over ranges the judge found.

In [6]:
index = json.load(Path("data/judge-input/index.json").open(encoding="utf-8"))
judged = {}
for f in sorted(Path("data/judge-output").glob("batch-*.jsonl")):
    for line in f.read_text(encoding="utf-8").splitlines():
        if line.strip():
            id_, *rest = json.loads(line)
            judged[id_] = tuple(rest)


def valid(outcome, start, end):
    if outcome != "range":
        return outcome in ("unparseable", "ambiguous")
    try:
        return bool(start or end) and all(date.fromisoformat(d.lstrip("-")) for d in (start, end) if d) and (not start or not end or start <= end)
    except ValueError:
        return False


expected = {}
for id_, (outcome, start, end, qualifier, note) in judged.items():
    if valid(outcome, start, end):
        for path in index[id_]["paths"]:
            expected[(id_, path)] = bounds(start, end) if outcome == "range" else (None, None)

got = {name: {(id_, path): results[name][(path, index[id_]["text"])] for id_, path in expected} for name in PARSERS}
weight = lambda key: occurrences[(key[1], index[key[0]]["text"])]

print(f"{len(judged)} judged, {len(judged) - len({k[0] for k in expected})} dropped as invalid\n")
print(f"{'parser':8} {'correct':>8} {'wrong':>6}   {'accuracy':>9} {'precision':>10} {'recall':>7}   {'weighted acc':>12} {'prec':>6} {'recall':>7}")
for name in PARSERS:
    line = f"{name:8}"
    for w in (lambda key: 1, weight):
        total = sum(w(k) for k in expected)
        correct = sum(w(k) for k in expected if got[name][k] == expected[k])
        produced = sum(w(k) for k in expected if got[name][k] != (None, None))
        judge_ranges = sum(w(k) for k in expected if expected[k] != (None, None))
        correct_ranges = sum(w(k) for k in expected if got[name][k] == expected[k] != (None, None))
        cells = (correct / total, correct_ranges / produced, correct_ranges / judge_ranges)
        line += f" {correct:>8} {total - correct:>6}   {cells[0]:9.1%} {cells[1]:10.1%} {cells[2]:7.1%}" if w(next(iter(expected))) == 1 and w is not weight else f"   {cells[0]:12.1%} {cells[1]:6.1%} {cells[2]:7.1%}"
    print(line)

300 judged, 0 dropped as invalid

parser    correct  wrong    accuracy  precision  recall   weighted acc   prec  recall
python        155    145       51.7%      53.7%   51.7%          77.9%  79.6%   77.9%
scala         247     53       82.3%      95.7%   82.4%          97.6%  99.7%   97.7%
new           282     18       94.0%      94.9%   94.9%          99.5%  99.6%   99.6%


In [7]:
PARSER = "new"
LIMIT = 40

for key in sorted((k for k in expected if got[PARSER][k] != expected[k]), key=lambda k: -weight(k))[:LIMIT]:
    id_, path = key
    outcome, start, end, qualifier, note = judged[id_]
    print(f"{index[id_]['text']!r}  [{path}, {weight(key)}x]")
    print(f"    judge   {show(expected[key]) if outcome == 'range' else outcome}{'  ' + repr(qualifier) if qualifier and qualifier != 'exact' else ''}{'  ' + note if note else ''}")
    print(f"    {PARSER:7} {show(got[PARSER][key])}")

'An. Dom. 1558. The vii. daye of October.'  [production, 2x]
    judge   1558-10-07 .. 1558-10-07  roman numeral converted
    new     1558-01-01 .. 1558-12-31
'1742/3.'  [production, 2x]
    judge   1742-01-01 .. 1743-12-31
    new     1742-01-01 .. 1742-12-31
'18--?]'  [production, 2x]
    judge   1800-01-01 .. 1899-12-31
    new     no range
'M.DCC.LXXXV.'  [production, 2x]
    judge   1785-01-01 .. 1785-12-31  roman numeral converted
    new     no range
'Printed in Decembe, 1782.'  [production, 2x]
    judge   1782-12-01 .. 1782-12-31
    new     1782-01-01 .. 1782-12-31
'First printed in the year 1767. Re-Printed in 1777.'  [production, 1x]
    judge   ambiguous  two separate dates, 1767 and 1777, not a range
    new     1767-01-01 .. 1777-12-31
'70-638.'  [subject, 1x]
    judge   0070-01-01 .. 0638-12-31
    new     no range
'Taishō 14 [1925]-'  [production, 1x]
    judge   1925-01-01 .. open  'after'
    new     1925-01-01 .. 1925-12-31
'between 1990 and 1999?-'  [production, 